In [ ]:
import cv2
import numpy as np
import time
from djitellopy import tello

drone = tello.Tello()
drone.connect()

try:
    print(f"Battery = {drone.get_battery()}")
except BaseException:
    print(f"Battery = 100 %")  

drone.streamon()
time.sleep(2)

FRAME_WIDTH = 360
FRAME_HEIGHT = 240
CENTER_X = FRAME_WIDTH // 2

MIN_AREA = 6200  
MAX_AREA = 6800  

FORWARD_BACKWARD_SPEED_SCALE = 0.005 
ROTATION_SPEED_SCALE = 0.4

drone.takeoff()
drone.move_up(40) 

while True:
    # 1. Capture the camera frame
    frame = drone.get_frame_read().frame
    frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
    
    face_detected = True 
    x, y, w, h = 130, 70, 80, 80 

    fb_velocity = 0
    yaw_velocity = 0

    if face_detected:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
        
        face_center_x = x + (w // 2)
        face_area = w * h
        
        if face_area < MIN_AREA:
            error_area = MIN_AREA - face_area
            fb_velocity = int(error_area * FORWARD_BACKWARD_SPEED_SCALE)
            # Cap speed safely
            fb_velocity = min(fb_velocity, 25) 
            cv2.putText(frame, "STATUS: Too Far - Moving Forward", (10, 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
            
        elif face_area > MAX_AREA:
            error_area = face_area - MAX_AREA
            fb_velocity = -int(error_area * FORWARD_BACKWARD_SPEED_SCALE)
            # Cap speed safely
            fb_velocity = max(fb_velocity, -25)
            cv2.putText(frame, "Too Close, Moving Back", (10, 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
            
        else:
            fb_velocity = 0
            cv2.putText(frame, "Perfet Distance", (10, 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

        center_error_x = face_center_x - CENTER_X
        yaw_velocity = int(center_error_x * ROTATION_SPEED_SCALE)
        
        cv2.putText(frame, f"Area : {face_area} px", (10, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

    else:
        cv2.putText(frame, "Searching for Face ...", (10, 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 100, 0), 1)

    drone.send_rc_control(left_right_velocity=0, 
                          up_down_velocity=0, 
                          forward_backward_velocity=fb_velocity, 
                          yaw_velocity=yaw_velocity)

    cv2.imshow("Autonomous Face Surveillance", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        drone.send_rc_control(0, 0, 0, 0)
        drone.land()
        break

drone.streamoff()
cv2.destroyAllWindows()
